In [1]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import average_precision_score

import pandas as pd
import numpy as np
import shutil
import csv

In [2]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
### Directories
project_root = Path.cwd()

summary_ground_truth_directory = project_root / Path("eval/gt_summary")
summary_predicted_directory = project_root / Path("eval/pred_summary")
frames_ground_truth_directory = project_root / Path("eval/gt_frames")
frames_predicted_directory = project_root / Path("eval/pred_frames")
result_directory = project_root / Path("eval/result")

summary_ground_truth_directory.mkdir(parents=True, exist_ok=True)
summary_predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [4]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [5]:
def calculate_prec_rec_f1(results_df, total_preds, threshold=0.5):
    tp_rows = results_df[results_df['tiou'] >= threshold]
    tp = len(tp_rows)
    matched_preds = len(set(idx for indices in tp_rows['pred_indices'] for idx in indices))
    fp = total_preds - matched_preds
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / len(results_df) if len(results_df) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_tiou(truth_csv_path, predicted_csv_path):
    import pandas as pd
    import numpy as np

    truth_df = pd.read_csv(truth_csv_path)
    pred_df = pd.read_csv(predicted_csv_path)
    matches = []

    for (hum_id, obj_id), truth_group in truth_df.groupby(['human_id', 'object_id']):
        pred_group = pred_df[(pred_df['human_id'] == hum_id) & (pred_df['object_id'] == obj_id)]
        
        if pred_group.empty:
            for t_idx, t_row in truth_group.iterrows():
                t_start = t_row['frame_start']
                t_end = t_row['frame_end']
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
            continue
            
        for t_idx, t_row in truth_group.iterrows():
            t_start = t_row['frame_start']
            t_end = t_row['frame_end']
            
            p_starts = pred_group['frame_start'].values
            p_ends = pred_group['frame_end'].values
            
            overlap_starts = np.maximum(t_start, p_starts)
            overlap_ends = np.minimum(t_end, p_ends)
            overlap_durations = np.maximum(0, (overlap_ends - overlap_starts) + 1)
            
            valid_mask = overlap_durations > 0
            
            if not np.any(valid_mask):
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
                continue
                
            v_p_starts = p_starts[valid_mask]
            v_p_ends = p_ends[valid_mask]
            v_overlap_durations = overlap_durations[valid_mask]
            
            intersection = np.sum(v_overlap_durations)
            
            t_duration = (t_end - t_start) + 1
            p_duration = np.sum((v_p_ends - v_p_starts) + 1)
            union = t_duration + p_duration - intersection
            
            tiou = intersection / union if union > 0 else 0
            
            t_frames = f"{t_start} - {t_end}"
            p_frames = [f"{s} - {e}" for s, e in zip(v_p_starts, v_p_ends)]
            p_indices = pred_group.index[valid_mask].tolist()
            
            matches.append((hum_id, obj_id, t_idx, p_indices, t_frames, p_frames, intersection, union, tiou))

    result_df = pd.DataFrame(matches, columns=['human_id', 'object_id', 'truth_idx', 'pred_indices', 'truth_frames', 'pred_frames', 'intersection', 'union', 'tiou'])
    return result_df

In [6]:
def evaluate_frame_level_score(truth_csv_path, predicted_csv_path):
    import pandas as pd
    t = set(map(tuple, pd.read_csv(truth_csv_path)[['frame_index', 'human_id', 'object_id']].values))
    p = set(map(tuple, pd.read_csv(predicted_csv_path)[['frame_index', 'human_id', 'object_id']].values))
    tp = len(t & p)
    precision = tp / len(p) if p else 0.0
    recall = tp / len(t) if t else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

In [7]:
def convert_truth_to_frames(csv_path, video_stride):
    df = pd.read_csv(csv_path)
    frame_interactions = [["frame_index", "human_id", "object_id", "distance"]]
    for _, row in df.iterrows():
        start = int(row["frame_start"])
        end = int(row["frame_end"])
        aligned_start = start + (video_stride - start % video_stride) % video_stride
        for i in range(aligned_start, end, video_stride):
            frame_interactions.append([i, int(row["human_id"]), int(row["object_id"]), row["distance"]])
    return frame_interactions

In [8]:
video_name = "vid01"
gt_csv_file = summary_ground_truth_directory / (video_name + "_true_summary.csv")
pred_csv_file = summary_predicted_directory / (video_name + "_pred_summary.csv")

result_df = evaluate_tiou(gt_csv_file, pred_csv_file)

total_preds = len(pd.read_csv(pred_csv_file))
tp_rows = result_df[result_df['tiou'] >= 0.5]
precision, recall, f1 = calculate_prec_rec_f1(result_df, total_preds, 0.5)

logs = []
logs.append(result_df.drop(columns=['truth_idx', 'pred_indices']).to_string())
logs.append(f"Total Predictions: {total_preds}")
logs.append(f"Mean TIOU: {result_df['tiou'].mean()}")
logs.append(f"Precision: {precision}")
logs.append(f"Recall: {recall}")
logs.append(f"F1: {f1}")

for log in logs:
    print(log)

with open(result_directory / f"{video_name}_result_summary", "w") as log_file:
    log_file.writelines("\n".join(logs))


   human_id  object_id truth_frames                                           pred_frames  intersection  union      tiou
0         1          4     0 - 3920                                 [0 - 100, 165 - 3920]          3857   3921  0.983678
1         6          9     76 - 575                                           [150 - 550]           401    500  0.802000
2         6         13    565 - 615                                           [565 - 615]            51     51  1.000000
3         6         17    659 - 810                                           [645 - 805]           147    166  0.885542
4         6         17    865 - 980                                           [865 - 980]           116    116  1.000000
5         6         17  1105 - 1235                                         [1105 - 1235]           131    131  1.000000
6         6         21    810 - 860                                           [820 - 860]            41     51  0.803922
7         6         28  1025 - 1

In [9]:
video_name = "test"
video_stride = 5
gt_csv_file = summary_ground_truth_directory / (video_name + "_true_summary.csv")

frames = convert_truth_to_frames(gt_csv_file, video_stride)
frames_flattened = [f"{f_id},{h_id},{o_id},{d}" for (f_id, h_id, o_id, d) in frames]
with open(frames_ground_truth_directory / f"{video_name}_pred_frames.csv", "w") as frames_truth_csv:
    frames_truth_csv.writelines("\n".join(frames_flattened))

In [10]:
video_name = "vid01"
gt_csv_file = frames_ground_truth_directory / (video_name + "_true_frames.csv")
pred_csv_file = frames_predicted_directory / (video_name + "_pred_frames.csv")

precision, recall, f1 = evaluate_frame_level_score(gt_csv_file, pred_csv_file)

logs = []
logs.append(f"Precision: {precision}")
logs.append(f"Recall: {recall}")
logs.append(f"F1: {f1}")

for log in logs:
    print(log)

with open(result_directory / f"{video_name}_result_frames", "w") as log_file:
    log_file.writelines("\n".join(logs))

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Gabriel_Files\\Programming_Files\\School\\Thesis\\src\\eval\\gt_frames\\vid01_true_frames.csv'